# C06 · SAR scene metadata for C2 (incidence angle, look direction, orbit)

BRIGHT ships 8-bit, terrain-corrected tiles **without** the scene metadata. C2 needs the viewing
geometry of each event's source scene, so this notebook finds it in the providers' public STAC
catalogs (Capella and Umbra open data, CC BY 4.0, no sign-in) and matches scenes to events by
area and date. The rule is fixed in `src/damagesar_bench/c2_metadata.py` (tested) and was set
before looking at any results.

* Only SAR **metadata** is read: no damage labels, so confirmatory events are fine here.
* Needs `GH_TOKEN` (to install the repo). No GPU needed: pick a CPU runtime.
* Output: `c2_sar_metadata_by_event.csv` (one row per event, with status geometry-ready /
  match-only / dilated-only) and `c2_sar_candidates.csv` (every candidate scene), saved to the run
  folder and to `01_data/external/c2_sar_metadata/`.

In [ ]:
NOTEBOOK_NAME = "c06_sar_scene_metadata"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# BRIGHT post-event SAR only (~2 GB). Used just for the tile footprints of each event.
import os, zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Event areas: union of tile bounds in WGS84
import collections, rasterio
from rasterio.warp import transform_bounds
from damagesar_bench import c2_metadata as c2
from damagesar_bench.data.events import event_key_from_filename

boxes = collections.defaultdict(list)
for f in sorted(os.listdir(post)):
    if not f.endswith(".tif"):
        continue
    with rasterio.open(f"{post}/{f}") as ds:
        b = transform_bounds(ds.crs, "EPSG:4326", *ds.bounds)
    boxes[event_key_from_filename(f)].append(c2.BBox.of(b))
AREAS = {e: c2.union_bbox(bs) for e, bs in boxes.items()}
for e, a in sorted(AREAS.items()):
    print(f"{e:<9} {len(boxes[e]):>5} tiles  W{a.west:.4f} S{a.south:.4f} E{a.east:.4f} N{a.north:.4f}")
assert set(AREAS) == set(c2.EVENT_DATES), set(AREAS) ^ set(c2.EVENT_DATES)

In [ ]:
# Walk the two static STAC catalogs (public S3, no sign-in). Umbra's is large, so its
# sub-catalogs are pruned to the years of the Umbra events (2023, 2024).
import re, time, requests
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

CATALOGS = {
    "capella": ("https://capella-open-data.s3.us-west-2.amazonaws.com/stac/catalog.json", None),
    "umbra": ("https://s3.us-west-2.amazonaws.com/umbra-open-data-catalog/stac/catalog.json", {"2023", "2024"}),
}
S = requests.Session()
def get(url):
    for i in range(3):
        try:
            r = S.get(url, timeout=30); r.raise_for_status(); return r.json()
        except Exception as ex:
            err = ex; time.sleep(1 + i)
    print("  failed:", url, err); return None

def keep(href, years):
    if years is None:
        return True
    found = set(re.findall(r"(?<!\d)(20\d\d)(?!\d)", href))
    return not found or bool(found & years)

def walk(provider, root, years, budget_s=1500):
    t0, items, seen = time.time(), [], set()
    frontier = [root]
    first = get(root)
    print(provider, "top-level children:", [l.get("title") or l["href"] for l in first.get("links", []) if l.get("rel") == "child"][:40])
    with ThreadPoolExecutor(32) as pool:
        while frontier and time.time() - t0 < budget_s:
            batch = [u for u in dict.fromkeys(frontier) if u not in seen]; seen.update(batch); frontier = []
            for url, obj in zip(batch, pool.map(get, batch)):
                if not obj:
                    continue
                if obj.get("type") == "Feature":
                    obj["provider"], obj["href"] = provider, url
                    items.append(obj); continue
                for l in obj.get("links", []):
                    if l.get("rel") in ("child", "item"):
                        h = urljoin(url, l["href"])
                        if keep(h, years):
                            frontier.append(h)
            print(f"  {provider}: {len(items)} items, {len(frontier)} queued, {time.time()-t0:.0f}s")
    if frontier:
        print(f"  WARNING {provider}: time budget hit with {len(frontier)} links unvisited")
    return items

ITEMS = {p: walk(p, url, yrs) for p, (url, yrs) in CATALOGS.items()}

In [ ]:
# Which property names exist? (so FIELD_KEYS can be checked against the real metadata)
keys = collections.Counter(k for its in ITEMS.values() for it in its for k in it.get("properties", {}))
geo_like = {k: v for k, v in keys.items() if re.search(r"angle|direction|orbit|look|incid|mode|squint|azimuth|layover|platform|product|resolution|gsd|pass", k, re.I)}
for k, v in sorted(geo_like.items()):
    print(f"{v:>6}  {k}")

In [ ]:
# Match scenes to events and write the tables
import pandas as pd, json
allitems = list({(it["provider"], it.get("id")): it for its in ITEMS.values() for it in its}.values())  # de-duplicated
cand_rows, event_rows = [], []
for e in c2.EVENT_DATES:
    cands = c2.match_event(e, AREAS[e], allitems)
    cand_rows += cands
    event_rows.append(c2.summarise_event(e, cands))
cands_df, events_df = pd.DataFrame(cand_rows), pd.DataFrame(event_rows)

OUTDIR = f"{PROJECT}/01_data/external/c2_sar_metadata"
os.makedirs(OUTDIR, exist_ok=True)
for d in (RUN, OUTDIR):
    cands_df.to_csv(f"{d}/c2_sar_candidates.csv", index=False)
    events_df.to_csv(f"{d}/c2_sar_metadata_by_event.csv", index=False)
# raw STAC JSON of every candidate, for provenance
ids = set(cands_df["item_id"]) if len(cands_df) else set()
with open(f"{OUTDIR}/c2_candidate_items.json", "w") as fh:
    json.dump([it for it in allitems if it.get("id") in ids], fh)

cols = ["event", "status", "n_candidates", "n_good_candidates", "geometry_consistent", "best_provider",
        "best_date", "best_days_after", "best_coverage", "best_incidence_deg", "best_look_direction",
        "best_orbit_state", "best_mode", "incidence_range_good"]
display(events_df[cols])
print(events_df["status"].value_counts().to_string())

**How to read it.** `geometry-ready` events can use projected layover/shadow masks in C2;
`match-only` need the missing field from the scene's own metadata file (the `href` / extended
metadata JSON); `dilated-only` events use the dilated-footprint fallback. `geometry_consistent = False`
means good candidate scenes disagree (different look/orbit or > 5° incidence): check which scene
BRIGHT used (visual match of a tile against the scene) before trusting the best row.
Record the outcome in `EXPERIMENT_LOG.md`.

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/checks/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))